In [ ]:
import tensorflow as tf
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

class HybridTumorDetector:
    def __init__(self, generator=None, discriminator=None, latent_dim=100):
        """
        Initialize the hybrid tumor detector with both GAN and ResNet components
        """
        self.latent_dim = latent_dim
        self.scaler = StandardScaler()
        self.class_thresholds = {}
        self.threshold = None
        
        # GAN components
        self.generator = generator
        self.discriminator = discriminator
        self.feature_model = None
        
        # ResNet classifier
        self.classifier = self.resnet_classifier()
        
        # Initialize GAN feature extractor if GAN components are provided
        if generator is not None and discriminator is not None:
            self._initialize_gan_feature_extractor()

    def _initialize_gan_feature_extractor(self):
        """Initialize GAN feature extractor"""
        try:
            # Initialize with a dummy input to build the model
            dummy_image = tf.zeros((1, 256, 256, 1), dtype=tf.float32)
            _ = self.discriminator(dummy_image, training=False)
            
            # Find the feature extraction layer (second to last layer)
            feature_layer = None
            for layer in reversed(self.discriminator.layers):
                if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
                    if layer != self.discriminator.layers[-1]:
                        feature_layer = layer
                        break
            
            if feature_layer is None:
                raise ValueError("Could not find appropriate feature layer in discriminator")
            
            # Create feature extraction model
            input_layer = tf.keras.Input(shape=(256, 256, 1))
            x = input_layer
            
            # Build up to the feature layer
            for layer in self.discriminator.layers:
                x = layer(x)
                if layer == feature_layer:
                    break
            
            self.feature_model = tf.keras.Model(inputs=input_layer, outputs=x)
            print("GAN feature extractor initialized successfully")
            
        except Exception as e:
            print(f"Error initializing GAN feature extractor: {str(e)}")
            raise

    def compute_anomaly_score(self, image):
        """Compute GAN-based anomaly score"""
        try:
            if self.generator is None or self.feature_model is None:
                raise ValueError("GAN components not initialized")

            image = tf.cast(image, tf.float32)
            if len(image.shape) == 3:
                image = image[np.newaxis, ...]

            # Generate multiple samples and find best reconstruction
            num_samples = 10
            latent_vectors = tf.random.normal((num_samples, self.latent_dim))
            reconstructions = self.generator(latent_vectors, training=False)
            
            # Compute reconstruction errors
            reconstruction_errors = tf.reduce_mean(tf.square(
                reconstructions - tf.repeat(image, num_samples, axis=0)
            ), axis=[1, 2, 3])

            # Get best reconstruction
            best_idx = tf.argmin(reconstruction_errors)
            best_reconstruction = reconstructions[best_idx]
            reconstruction_error = reconstruction_errors[best_idx]

            # Compute feature-level error
            real_feat = self.feature_model(image, training=False)
            fake_feat = self.feature_model(best_reconstruction[np.newaxis, ...], training=False)
            feature_error = tf.reduce_mean(tf.square(real_feat - fake_feat))

            # Combine errors with weights
            anomaly_score = float(0.5 * reconstruction_error + 0.5 * feature_error)
            return anomaly_score, best_reconstruction

        except Exception as e:
            print(f"Error computing anomaly score: {str(e)}")
            return None, None

    def set_threshold(self, normal_data_path, percentile=95):
        """Set the anomaly threshold based on normal data"""
        if self.generator is None or self.feature_model is None:
            print("Warning: GAN components not initialized. Threshold setting skipped.")
            return False

        try:
            anomaly_scores = []
            
            print("Calculating anomaly scores for normal data...")
            for img_path in tqdm(list(Path(normal_data_path).glob('*.[jp][pn][g]'))):
                image = self.preprocess_image(img_path)
                if image is None:
                    continue
                    
                anomaly_score, _ = self.compute_anomaly_score(image)
                if anomaly_score is not None:
                    anomaly_scores.append(anomaly_score)
            
            if not anomaly_scores:
                raise ValueError("No valid anomaly scores computed from normal data")
                
            self.threshold = np.percentile(anomaly_scores, percentile)
            print(f"Anomaly detection threshold set to: {self.threshold}")
            return True

        except Exception as e:
            print(f"Error setting threshold: {str(e)}")
            return False

    def load_gan_models(self, generator_path, discriminator_path):
        """Load pre-trained GAN models"""
        try:
            self.generator = tf.keras.models.load_model(generator_path)
            self.discriminator = tf.keras.models.load_model(discriminator_path)
            self._initialize_gan_feature_extractor()
            print("GAN models loaded successfully")
            return True
        except Exception as e:
            print(f"Error loading GAN models: {str(e)}")
            return False

    def preprocess_image(self, image_path, target_size=(256, 256)):
        """Preprocess image for both ResNet and GAN"""
        try:
            # Read image in grayscale
            img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError(f"Could not read image at {image_path}")

            # Basic preprocessing
            img = cv2.resize(img, target_size)
            img = img.astype(np.float32)
            img = img / 255.0  # Normalize to [0,1]
            
            # Return single-channel image
            if len(img.shape) == 2:
                img = np.expand_dims(img, axis=-1)
            return img
        
        except Exception as e:
            print(f"Error preprocessing {image_path}: {str(e)}")
            traceback.print_exc()
            return None

    def resnet_classifier(self):
        """Build a ResNet-based classifier for brain tumor classification"""
        base_model = tf.keras.applications.ResNet50V2(
            include_top=False,
            weights='imagenet',
            input_shape=(256, 256, 3),
            pooling='avg'
        )
        
        # Freeze early layers for transfer learning
        for layer in base_model.layers[:100]:
            layer.trainable = False
        
        # Build the complete model
        inputs = tf.keras.Input(shape=(256, 256, 1))
        
        # Convert single channel to RGB - using a simpler approach
        x = inputs
        x = tf.keras.layers.Conv2D(3, (1, 1))(x)  # Use 1x1 convolution to control channel dimension
        
        # Manual rescaling
        x = tf.keras.layers.Lambda(lambda x: x * 2.0 - 1.0)(x)
        
        # Pass through ResNet
        x = base_model(x)
        
        # Add custom classification head
        x = tf.keras.layers.Dense(512, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.3)(x)
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
            loss='categorical_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(), tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
        )
        
        return model

    def train_classifier(self, train_data_path, validation_split=0.2, epochs=50, batch_size=16):
        """Train the ResNet classifier with MRI-specific augmentation and monitoring"""
        try:
            class_names = ['notumor', 'glioma', 'meningioma', 'pituitary']
            images = []
            labels = []

            # Load and preprocess images
            print("Loading training data...")
            for idx, class_name in enumerate(class_names):
                class_path = Path(train_data_path) / class_name
                if not class_path.exists():
                    raise ValueError(f"Class path not found: {class_path}")

                for img_path in tqdm(list(class_path.glob('*.[jp][pn][g]'))):
                    image = self.preprocess_image(img_path)
                    if image is not None:
                        images.append(image)
                        labels.append(idx)

            images = np.array(images)
            labels = tf.keras.utils.to_categorical(labels, num_classes=4)

            # Split data
            indices = np.arange(len(images))
            np.random.shuffle(indices)
            split_idx = int(len(images) * (1 - validation_split))
            train_indices = indices[:split_idx]
            val_indices = indices[split_idx:]

            # Define MRI-specific augmentation layers
            class MRIIntensityAugmentation(tf.keras.layers.Layer):
                def __init__(self, intensity_range=0.1, gamma_range=(0.9, 1.1), **kwargs):
                    super().__init__(**kwargs)
                    self.intensity_range = intensity_range
                    self.gamma_range = gamma_range

                def call(self, inputs, training=True):
                    if not training:
                        return inputs

                    # Random intensity scaling
                    intensity_factor = tf.random.uniform([], 
                                                       1 - self.intensity_range, 
                                                       1 + self.intensity_range)
                    x = inputs * intensity_factor

                    # Random gamma adjustment (simulates MRI contrast variations)
                    gamma = tf.random.uniform([], self.gamma_range[0], self.gamma_range[1])
                    x = tf.pow(x, gamma)

                    # Ensure values stay in valid range
                    return tf.clip_by_value(x, 0, 1)

            class MRINoiseAugmentation(tf.keras.layers.Layer):
                def __init__(self, rician_noise_scale=0.05, **kwargs):
                    super().__init__(**kwargs)
                    self.noise_scale = rician_noise_scale

                def call(self, inputs, training=True):
                    if not training:
                        return inputs

                    # Rician noise (common in MRI)
                    shape = tf.shape(inputs)
                    real_noise = tf.random.normal(shape, mean=0, stddev=self.noise_scale)
                    imag_noise = tf.random.normal(shape, mean=0, stddev=self.noise_scale)
                    rician_noise = tf.sqrt(tf.square(real_noise + inputs) + tf.square(imag_noise))

                    return tf.clip_by_value(rician_noise, 0, 1)

            augmentation_layers = [
                # Minimal rotation (only ±10 degrees max to account for patient positioning)
                tf.keras.layers.RandomRotation(0.028),  # 0.028 radians ≈ 10 degrees

                # Very small zoom to account for minor scale variations in MRI
                tf.keras.layers.RandomZoom(
                    height_factor=(-0.05, 0.05),
                    width_factor=(-0.05, 0.05)
                ),

                # Custom MRI-specific augmentations
                MRIIntensityAugmentation(
                    intensity_range=0.1,    # Controls brightness variation
                    gamma_range=(0.9, 1.1)  # Controls contrast variation
                ),

                # MRI-specific noise
                MRINoiseAugmentation(rician_noise_scale=0.03)
            ]

            def apply_augmentation(x, y):
                for layer in augmentation_layers:
                    x = layer(x, training=True)
                return x, y

            # Create datasets with augmentation
            train_dataset = tf.data.Dataset.from_tensor_slices((
                images[train_indices], labels[train_indices]
            )).shuffle(1000).map(
                apply_augmentation,
                num_parallel_calls=tf.data.AUTOTUNE
            ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

            val_dataset = tf.data.Dataset.from_tensor_slices((
                images[val_indices], labels[val_indices]
            )).batch(batch_size).prefetch(tf.data.AUTOTUNE)

            # Advanced callbacks
            callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_auc',
                    patience=15,
                    restore_best_weights=True,
                    mode='max'
                ),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.2,
                    patience=7,
                    min_lr=1e-7
                ),
                # Add validation image logging callback
                tf.keras.callbacks.ModelCheckpoint(
                    'best_model.keras',
                    monitor='val_auc',
                    mode='max',
                    save_best_only=True,
                    save_weights_only=False
                )
            ]

            # Training with class weights to handle imbalance
            class_weights = {
                i: len(images) / (len(class_names) * np.sum(labels[:, i]))
                for i in range(len(class_names))
            }

            # Perform training
            history = self.classifier.fit(
                train_dataset,
                validation_data=val_dataset,
                epochs=epochs,
                callbacks=callbacks,
                class_weight=class_weights
            )

            self.training_history = history.history
            return True

        except Exception as e:
            print(f"Error in ResNet training: {str(e)}")
            return False

    def save_models(self, save_dir):
        """
        Save all model components and metadata in deployment-friendly formats
    
        Args:
        save_dir (str): Directory to save model files
        """
        save_dir = Path(save_dir)
        save_dir.mkdir(exist_ok=True)
    
        try:
            # Save main classifier model in .h5 format
            self.classifier.save(save_dir / 'resnet_classifier.h5')
        
            # Save GAN components if available
            if self.generator is not None:
                self.generator.save(save_dir / 'generator.h5')
            if self.discriminator is not None:
                self.discriminator.save(save_dir / 'discriminator.h5')
            if self.feature_model is not None:
                self.feature_model.save(save_dir / 'feature_extractor.h5')
            
        # Save threshold and other metadata
            metadata = {
                'threshold': self.threshold,
                'model_config': {
                    'input_shape': self.classifier.input_shape,
                    'num_classes': self.classifier.output_shape[-1],
                    'latent_dim': self.latent_dim
                },
                'class_names': ['notumor', 'glioma', 'meningioma', 'pituitary']
            }
            np.save(save_dir / 'metadata.npy', metadata)
        
        # Save scaler if used
            if hasattr(self, 'scaler') and self.scaler is not None:
                from joblib import dump
                dump(self.scaler, save_dir / 'scaler.joblib')
            
        # Save training history if available
            if hasattr(self, 'training_history'):
                np.save(save_dir / 'training_history.npy', self.training_history)
            
            # Save model architecture as JSON for reference
            model_json = self.classifier.to_json()
            with open(save_dir / 'model_architecture.json', 'w') as f:
                f.write(model_json)
            
        # Save a deployment config file
            deploy_config = {
                'model_files': {
                    'classifier': 'resnet_classifier.h5',
                    'generator': 'generator.h5' if self.generator is not None else None,
                    'discriminator': 'discriminator.h5' if self.discriminator is not None else None,
                    'feature_extractor': 'feature_extractor.h5' if self.feature_model is not None else None
                },
                'metadata_file': 'metadata.npy',
                'scaler_file': 'scaler.joblib' if hasattr(self, 'scaler') and self.scaler is not None else None,
                'architecture_file': 'model_architecture.json',
                'training_history': 'training_history.npy' if hasattr(self, 'training_history') else None
            }
        
            with open(save_dir / 'deploy_config.json', 'w') as f:
                json.dump(deploy_config, f, indent=4)
            
            print(f"All model components saved successfully to {save_dir}")
            return True
        
        except Exception as e:
            print(f"Error saving models: {str(e)}")
            return False

    def load_models(self, load_dir):
        """
        Load all model components and metadata
    
        Args:
            load_dir (str): Directory containing saved model files
        """
        load_dir = Path(load_dir)
        try:
        # Load deploy config
            with open(load_dir / 'deploy_config.json', 'r') as f:
                deploy_config = json.load(f)
            
        # Load main classifier
            self.classifier = tf.keras.models.load_model(
                load_dir / deploy_config['model_files']['classifier']
            )
        
        # Load GAN components if available
            if deploy_config['model_files']['generator']:
                self.generator = tf.keras.models.load_model(
                    load_dir / deploy_config['model_files']['generator']
                )
            if deploy_config['model_files']['discriminator']:
                self.discriminator = tf.keras.models.load_model(
                    load_dir / deploy_config['model_files']['discriminator']
                )
            if deploy_config['model_files']['feature_extractor']:
                self.feature_model = tf.keras.models.load_model(
                    load_dir / deploy_config['model_files']['feature_extractor']
                )
            
        # Load metadata
            metadata = np.load(load_dir / deploy_config['metadata_file'], allow_pickle=True).item()
            self.threshold = metadata['threshold']
            self.latent_dim = metadata['model_config']['latent_dim']
        
        # Load scaler if available
            if deploy_config['scaler_file']:
                from joblib import load
                self.scaler = load(load_dir / deploy_config['scaler_file'])
            
        # Load training history if available
            if deploy_config['training_history']:
                self.training_history = np.load(
                    load_dir / deploy_config['training_history']
                ).item()
            
            print(f"All model components loaded successfully from {load_dir}")
            return True
        
        except Exception as e:
            print(f"Error loading models: {str(e)}")
            traceback.print_exc()
            return False

    def plot_training_history(self):
        """Plot training history"""
        if not hasattr(self, 'training_history'):
            print("No training history available")
            return
        
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.training_history['accuracy'], label='Training Accuracy')
        plt.plot(self.training_history['val_accuracy'], label='Validation Accuracy')
        plt.title('Model Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(self.training_history['loss'], label='Training Loss')
        plt.plot(self.training_history['val_loss'], label='Validation Loss')
        plt.title('Model Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.tight_layout()
        plt.show()
    def evaluate(self, test_data_path):
        """Evaluate the hybrid detector"""
        if self.threshold is None:
            raise ValueError("Threshold not set. Call set_threshold() before evaluation")
            
        results = {
            'predictions': [],
            'true_labels': [],
            'confidences': []
        }
        
        classes = ['notumor', 'glioma', 'meningioma', 'pituitary']
        
        for class_name in classes:
            class_path = Path(test_data_path) / class_name
            if not class_path.exists():
                print(f"Warning: {class_path} not found")
                continue
            
            print(f"Evaluating {class_name} images...")
            for img_path in tqdm(list(class_path.glob('*.[jp][pn][g]'))):
                try:
                    image = self.preprocess_image(img_path)
                    if image is None:
                        continue
                    
                    prediction, confidences, _ = self.predict_hybrid(image)
                    if prediction is None:
                        continue
                    
                    results['predictions'].append(prediction)
                    results['true_labels'].append(class_name)
                    results['confidences'].append(confidences)
                    
                except Exception as e:
                    print(f"Error processing {img_path}: {str(e)}")
                    continue
        
        # Calculate metrics
        predictions = np.array(results['predictions'])
        true_labels = np.array(results['true_labels'])
        
        print("\nClassification Report:")
        print(classification_report(true_labels, predictions))
        
        cm = confusion_matrix(true_labels, predictions)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
        plt.title('Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()
        
        return results
    def predict_hybrid(self, image):
        """
        Make a prediction using both ResNet-based classification and GAN-based anomaly detection.
        """
        try:
            # Store original single-channel image for GAN
            original_image = image.copy()
            
            # Ensure input is single channel and has batch dimension
            if len(image.shape) == 2:
                image = np.expand_dims(image, axis=-1)
            if len(image.shape) == 3 and image.shape[-1] != 1:
                image = image[..., :1]
            if len(image.shape) == 3:
                image = np.expand_dims(image, axis=0)

            # Get ResNet prediction
            resnet_predictions = self.classifier.predict(image, verbose=0)
            confidence_scores = resnet_predictions[0]
            predicted_class_idx = np.argmax(confidence_scores)
            classes = ['notumor', 'glioma', 'meningioma', 'pituitary']
            predicted_class = classes[predicted_class_idx]

            # Compute anomaly score if GAN components are available
            anomaly_score = None
            if self.generator is not None and self.feature_model is not None:
                # Use original_image for GAN
                if len(original_image.shape) == 2:
                    original_image = np.expand_dims(original_image, axis=-1)
                if len(original_image.shape) == 3:
                    original_image = np.expand_dims(original_image, axis=0)
                
                anomaly_score, _ = self.compute_anomaly_score(original_image)

                # Apply threshold-based decision
                if self.threshold is not None and anomaly_score > self.threshold:
                    if predicted_class == 'notumor':
                        predicted_class = 'glioma'  # Default to most common type
                        confidence_scores = np.array([0.0, 0.5, 0.25, 0.25])

            return predicted_class, confidence_scores, anomaly_score

        except Exception as e:
            print(f"Error in hybrid prediction: {str(e)}")
            traceback.print_exc()
            return None, None, None


In [ ]:
def train_detector(train_data_path, generator_path=None, discriminator_path=None, save_dir='models'):
    """Train the hybrid detector"""
    detector = HybridTumorDetector()
    
    # Load GAN models if paths provided
    if generator_path and discriminator_path:
        if not detector.load_gan_models(generator_path, discriminator_path):
            print("Continuing without GAN components...")
    
    # Train classifier
    print("Training ResNet classifier...")
    if detector.train_classifier(train_data_path):
        print("ResNet training completed successfully")
        
        # Save trained models
        if detector.save_models(save_dir):
            print(f"Models saved to {save_dir}")
        
        return detector
    return None

# evaluation.py
def evaluate_detector(detector, test_data_path):
    """Evaluate the hybrid detector"""
    if detector.threshold is None:
        print("Warning: Threshold not set. Only CNN evaluation will be performed.")
    
    try:
        results = detector.evaluate(test_data_path)
        return results
    except Exception as e:
        print(f"Error in evaluation: {str(e)}")

In [ ]:
if __name__ == '__main__':
    # Configuration
    generator_path = "/kaggle/input/braingan2/generator_epoch_8500.h5"
    discriminator_path = "/kaggle/input/braingan2/discriminator_epoch_8500.h5"
    train_data_path = '/kaggle/input/brain-tumor-mri-dataset/Training'
    test_data_path = '/kaggle/input/brain-tumor-mri-dataset/Testing'
    normal_data_path = '/kaggle/input/brain-tumor-mri-dataset/Training/notumor'
    save_dir = '/kaggle/working'
    # Training phase
    detector = train_detector(
        train_data_path=train_data_path,
        generator_path=generator_path,
        discriminator_path=discriminator_path,
        save_dir='models'
    )
    
    if detector is not None:
        # Set threshold
        detector.set_threshold(normal_data_path)
        
        # Evaluation phase
        results = evaluate_detector(detector, test_data_path)
        
        if results is not None:
            print("Evaluation completed successfully")